# UWPR Contaminant Peak Analysis in NIST23 Data

Two-stage analysis:
1. **Stage 1 (Mass match)**: Find NIST23 peaks whose m/z matches a known UWPR contaminant within tolerance
2. **Stage 2 (Formula feasibility)**: Among mass-matched peaks, check if the contaminant formula could be a real fragment of the parent molecule (all element counts ≤ parent formula → plausibly real peak)

Peaks that match by mass but are **formula-infeasible** (contaminant formula has more of some element than the parent) are strong contaminant candidates.

In [ ]:
import ast
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src")))
from fragnnet.utils.formula_utils import parse_formula

## 1. Load UWPR / MaConDa contaminant lists

In [ ]:
DATA_DIR = Path("../data/umpr")

# --- MaConDa (has formula + mz + ion_mode + type) ---
maconda = pd.read_csv(
    DATA_DIR / "UWPR_CommonMassSpecContaminants(MaConDa).csv",
    skiprows=3,
    dtype=str,
    encoding="latin-1",
)
maconda.columns = maconda.columns.str.strip()
maconda["mz"] = pd.to_numeric(maconda["mz"], errors="coerce")
maconda["exact_adduct_mass"] = pd.to_numeric(maconda["exact_adduct_mass"], errors="coerce")
maconda = maconda.dropna(subset=["mz"])
maconda["ion_mode"] = maconda["ion_mode"].str.strip()
print(f"MaConDa entries: {len(maconda)}, ion modes: {maconda['ion_mode'].value_counts().to_dict()}")
maconda.head(3)

In [ ]:
# --- UWPR Positive file (Keller et al. 2008) ---
uwpr_pos_raw = pd.read_csv(
    DATA_DIR / "UWPR_CommonMassSpecContaminants(Positive).csv",
    skiprows=7,
    header=None,
    dtype=str,
    encoding="latin-1",
)
uwpr_pos = uwpr_pos_raw[[0, 1, 2, 3, 4]].copy()
uwpr_pos.columns = ["mz", "ion_type", "formula", "compound", "origin"]
uwpr_pos["mz"] = pd.to_numeric(uwpr_pos["mz"], errors="coerce")
uwpr_pos = uwpr_pos.dropna(subset=["mz"])
uwpr_pos["ion_mode"] = "POS"
print(f"UWPR Positive entries: {len(uwpr_pos)}")
uwpr_pos.head(3)

In [ ]:
# --- UWPR Negative file ---
uwpr_neg_raw = pd.read_csv(
    DATA_DIR / "UWPR_CommonMassSpecContaminants(Negative).csv",
    skiprows=7,
    header=None,
    dtype=str,
    encoding="latin-1",
)
uwpr_neg = uwpr_neg_raw[[0, 1, 2, 3, 4]].copy()
uwpr_neg.columns = ["mz", "ion_type", "formula", "compound", "origin"]
uwpr_neg["mz"] = pd.to_numeric(uwpr_neg["mz"], errors="coerce")
uwpr_neg = uwpr_neg.dropna(subset=["mz"])
uwpr_neg["ion_mode"] = "NEG"
print(f"UWPR Negative entries: {len(uwpr_neg)}")
uwpr_neg.head(3)

In [ ]:
# Build unified contaminant tables keyed by ion_mode
# MaConDa as primary (has formula info); UWPR Keller list fills in extra m/z values
maconda_pos = maconda[maconda["ion_mode"] == "POS"][
    ["mz", "formula", "name", "type_of_contaminant", "ion_form"]
].copy()
maconda_pos.columns = ["mz", "formula", "compound", "type", "ion_form"]
maconda_pos["source"] = "MaConDa"

maconda_neg = maconda[maconda["ion_mode"] == "NEG"][
    ["mz", "formula", "name", "type_of_contaminant", "ion_form"]
].copy()
maconda_neg.columns = ["mz", "formula", "compound", "type", "ion_form"]
maconda_neg["source"] = "MaConDa"

# UWPR Keller pos
uwpr_pos_slim = uwpr_pos[["mz", "formula", "compound", "ion_type"]].copy()
uwpr_pos_slim["type"] = "unknown"
uwpr_pos_slim["ion_form"] = uwpr_pos_slim["ion_type"]
uwpr_pos_slim["source"] = "UWPR_Keller"
uwpr_pos_slim = uwpr_pos_slim.drop(columns=["ion_type"])

# Merge and deduplicate by m/z (keep first, MaConDa preferred since it has formula)
cont_pos = pd.concat([maconda_pos, uwpr_pos_slim], ignore_index=True).drop_duplicates(subset=["mz"])
cont_neg = maconda_neg.drop_duplicates(subset=["mz"])

print(f"Unique contaminant m/z values — POS: {len(cont_pos)}, NEG: {len(cont_neg)}")

## 2. Load NIST23 processed data

In [ ]:
PROC_DIR = Path("../data/proc/nist23")

spec_df = pd.read_pickle(PROC_DIR / "spec_df.pkl")
mol_df = pd.read_pickle(PROC_DIR / "mol_df.pkl")

print(f"spec_df: {len(spec_df)} spectra, columns: {list(spec_df.columns)}")
print(f"mol_df:  {len(mol_df)} molecules, columns: {list(mol_df.columns)}")

In [ ]:
# Merge formula into spec_df
spec_mol = spec_df.merge(mol_df[["mol_id", "formula"]], on="mol_id", how="left")
print(f"ion_mode distribution: {spec_mol['ion_mode'].value_counts().to_dict()}")
spec_mol.head(2)

## 3. Parse peaks into (mz, intensity) arrays

Peaks are stored as Python list-of-tuples literals: `[(mz, intensity), ...]`

In [ ]:
def parse_peaks(peaks) -> np.ndarray:
    """Parse peaks stored as list of (mz, intensity) tuples into (N, 2) float32 array."""
    if peaks is None:
        return np.empty((0, 2), dtype=np.float32)
    if isinstance(peaks, str):
        try:
            peaks = ast.literal_eval(peaks)
        except Exception:
            return np.empty((0, 2), dtype=np.float32)
    if not peaks:
        return np.empty((0, 2), dtype=np.float32)
    return np.array(peaks, dtype=np.float32)


# Quick sanity check on the first spectrum
sample = spec_mol["peaks"].iloc[0]
arr = parse_peaks(sample)
print(f"Sample peaks type: {type(sample).__name__}, parsed shape: {arr.shape}")
print(f"First 3 peaks: {arr[:3]}")

In [ ]:
# Keep only P/N spectra, explode peaks into a flat table for vectorised matching
spec_pn = spec_mol[spec_mol["ion_mode"].isin(["P", "N"])].copy()

print("Parsing and exploding peaks...")
spec_pn["peak_mzs"] = spec_pn["peaks"].map(
    lambda p: parse_peaks(p)[:, 0].tolist() if parse_peaks(p).shape[0] > 0 else []
)

peaks_flat = (
    spec_pn[["spec_id", "mol_id", "formula", "ion_mode", "inst_type", "peak_mzs"]]
    .explode("peak_mzs")
    .dropna(subset=["peak_mzs"])
    .rename(columns={"peak_mzs": "peak_mz"})
)
peaks_flat["peak_mz"] = peaks_flat["peak_mz"].astype(np.float64)
print(f"Total peaks to scan: {len(peaks_flat):,}")

## 4. Formula feasibility check

In [ ]:
def formula_feasible(parent_formula: str, contaminant_formula: str) -> bool | None:
    """Return True if contaminant formula could be a subfragment of parent.

    A contaminant formula is feasible as a real fragment if every element count
    in the contaminant formula is <= the corresponding count in the parent.
    Allows ±4 H flexibility to account for rearrangements.

    Returns None if either formula cannot be parsed.
    """
    H_FLEX = 4
    try:
        p = parse_formula(parent_formula)
        c = parse_formula(contaminant_formula)
    except Exception:
        return None

    for elem, count in c.items():
        if elem == "H":
            if count > p.get("H", 0) + H_FLEX:
                return False
        else:
            if count > p.get(elem, 0):
                return False
    return True


assert formula_feasible("C6H12O6", "C2H4O") is True   # glucose → ethylene oxide fragment
assert formula_feasible("C6H12O6", "C7H6O") is False  # more C than parent
print("formula_feasible OK")

## 5. Stage 1: Mass-match peaks vs. UWPR list (vectorised)

Uses `pd.merge_asof` with `direction='nearest'` and `tolerance` for O(N log N) matching.

In [ ]:
MZ_TOL_DA = 0.05  # 50 mDa absolute tolerance

cont_pos_sorted = cont_pos.sort_values("mz").reset_index(drop=True)
cont_neg_sorted = cont_neg.sort_values("mz").reset_index(drop=True)


def match_mode(peaks_mode: pd.DataFrame, cont_sorted: pd.DataFrame, tol: float) -> pd.DataFrame:
    """Vectorised nearest-neighbour mass match using merge_asof."""
    left = peaks_mode.sort_values("peak_mz")
    right = cont_sorted[["mz", "formula", "compound", "type", "ion_form", "source"]].rename(
        columns={"mz": "cont_mz", "formula": "cont_formula", "compound": "cont_compound",
                 "type": "cont_type", "ion_form": "cont_ion_form", "source": "cont_source"}
    )
    merged = pd.merge_asof(
        left,
        right,
        left_on="peak_mz",
        right_on="cont_mz",
        direction="nearest",
        tolerance=tol,
    )
    hits = merged.dropna(subset=["cont_mz"]).copy()
    hits["delta_mz"] = hits["peak_mz"] - hits["cont_mz"]
    return hits


print("Matching positive mode peaks...")
hits_pos = match_mode(
    peaks_flat[peaks_flat["ion_mode"] == "P"],
    cont_pos_sorted,
    MZ_TOL_DA,
)
print(f"  POS hits: {len(hits_pos):,} in {hits_pos['spec_id'].nunique():,} spectra")

print("Matching negative mode peaks...")
hits_neg = match_mode(
    peaks_flat[peaks_flat["ion_mode"] == "N"],
    cont_neg_sorted,
    MZ_TOL_DA,
)
print(f"  NEG hits: {len(hits_neg):,} in {hits_neg['spec_id'].nunique():,} spectra")

hits_df = pd.concat([hits_pos, hits_neg], ignore_index=True)
print(f"\nTotal hit (peak, contaminant) pairs: {len(hits_df):,}")
print(f"Unique spectra with ≥1 mass hit:     {hits_df['spec_id'].nunique():,}")
print(f"Unique contaminant m/z matched:      {hits_df['cont_mz'].nunique():,}")

## 6. Stage 2: Formula feasibility filter

In [ ]:
def check_feasibility(row) -> str:
    parent_f = row["formula"]
    cont_f = row["cont_formula"]
    if not isinstance(cont_f, str) or not cont_f.strip() or cont_f.strip() == "nan":
        return "no_cont_formula"
    if not isinstance(parent_f, str) or not parent_f.strip():
        return "no_parent_formula"
    result = formula_feasible(parent_f, cont_f)
    if result is None:
        return "parse_error"
    return "feasible" if result else "infeasible"


hits_df["feasibility"] = hits_df.apply(check_feasibility, axis=1)
print(hits_df["feasibility"].value_counts())

## 7. Summary statistics

In [ ]:
total_spectra = len(spec_pn)
total_pos = (spec_pn["ion_mode"] == "P").sum()
total_neg = (spec_pn["ion_mode"] == "N").sum()

n_any_hit = hits_df["spec_id"].nunique()
n_infeasible = hits_df[hits_df["feasibility"] == "infeasible"]["spec_id"].nunique()
# Spectra where ALL hits are feasible (no definite contaminant signal)
per_spec_feas = hits_df.groupby("spec_id")["feasibility"].apply(
    lambda x: (x == "feasible").all()
)
n_feasible_only = per_spec_feas.sum()

print(f"Total NIST23 spectra (P+N):      {total_spectra:,}")
print(f"  Positive mode:                 {total_pos:,}")
print(f"  Negative mode:                 {total_neg:,}")
print()
print(f"Stage 1 — mass match (±{MZ_TOL_DA} Da):")
print(f"  Spectra with ≥1 UWPR hit:      {n_any_hit:,} ({100*n_any_hit/total_spectra:.1f}%)")
print(f"  Hit (peak, cont_mz) pairs:     {len(hits_df):,}")
print()
print("Stage 2 — formula feasibility breakdown:")
print(hits_df["feasibility"].value_counts().to_string())
print()
print(f"  Spectra with ≥1 infeasible hit (contaminant candidate):")
print(f"    {n_infeasible:,} ({100*n_infeasible/total_spectra:.1f}% of all P/N spectra)")
print(f"  Spectra where all hits are feasible (could be real peaks):")
print(f"    {n_feasible_only:,} ({100*n_feasible_only/total_spectra:.1f}% of all P/N spectra)")

In [ ]:
# Breakdown by instrument type
print("Spectra with ≥1 infeasible UWPR hit, by instrument type:")
infeasible_spec_ids = hits_df[hits_df["feasibility"] == "infeasible"]["spec_id"].unique()
inst_contam = spec_pn[spec_pn["spec_id"].isin(infeasible_spec_ids)]["inst_type"].value_counts()
inst_total = spec_pn["inst_type"].value_counts()
breakdown = pd.DataFrame({"contam_spectra": inst_contam, "total_spectra": inst_total}).fillna(0)
breakdown["pct"] = (100 * breakdown["contam_spectra"] / breakdown["total_spectra"]).round(1)
print(breakdown.sort_values("pct", ascending=False).to_string())

In [ ]:
# Top contaminant m/z values by frequency (all feasibility)
print("Top 20 most frequent UWPR mass matches:")
top_cont = (
    hits_df.groupby(["cont_mz", "cont_compound", "cont_type", "feasibility"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)
print(top_cont.to_string(index=False))

In [ ]:
# Top infeasible hits — strongest contaminant candidates
print("Top 20 infeasible (likely contaminant) UWPR peaks:")
top_infeasible = (
    hits_df[hits_df["feasibility"] == "infeasible"]
    .groupby(["cont_mz", "cont_compound", "cont_formula", "cont_type"])
    .agg(n_spectra=("spec_id", "nunique"))
    .reset_index()
    .sort_values("n_spectra", ascending=False)
    .head(20)
)
print(top_infeasible.to_string(index=False))

In [ ]:
# Feasible hits — could be real fragments that coincide with contaminant masses
print("Top 20 feasible (could be real fragment) UWPR masses:")
top_feasible = (
    hits_df[hits_df["feasibility"] == "feasible"]
    .groupby(["cont_mz", "cont_compound", "cont_formula", "cont_type"])
    .agg(n_spectra=("spec_id", "nunique"))
    .reset_index()
    .sort_values("n_spectra", ascending=False)
    .head(20)
)
print(top_feasible.to_string(index=False))

In [ ]:
# Save full results
out_path = Path("../data/umpr/nist23_uwpr_hits.csv")
hits_df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")